# Rowwise INT8 Quantization

Bot task id: `11_quantize_rowwise_int8`, deadline 06.12.2026 00:00 MSK. [Open in Colab](https://colab.research.google.com/github/PE51K/itmo-ai-talent-hub-efficient-models-course/blob/main/hw3/11_quantize_rowwise_int8.ipynb)

## Условие

Построчная (per-row) симметричная квантизация. Дан 2D тензор `X` формы `(N_ROWS, N_COLS)` (fp16/fp32, CUDA, row-major, contiguous). Вернуть `Q: int8` той же формы и вектор `absmaxs` длины `N_ROWS`.

Для каждой строки $r$:

$$a_r = \max_c |X[r, c]| \ \text{(без clamp)}, \qquad Q[r, c] = \operatorname{round}(127 \cdot X[r, c] / a_r)$$

**Требования**
- одно Triton-ядро, одна строка на программу; ширину строки расширяем до ближайшей степени двойки `P2` для редукции (хвост маскируем);
- `tl.load` / `tl.store` и маскирование по столбцам;
- обязательно `@triton.autotune` с несколькими конфигурациями (в сигнатуре их надо вписать самим);
- обёртка `quantize_rowwise(x)` возвращает `(q_int8, absmaxs)`, где `absmaxs.dtype == torch.float16` и `absmaxs.shape == (N_ROWS,)`.

Пока в `@triton.autotune()` нет конфигов, файл падает при импорте.

## Setup

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import importlib.util

import torch
import triton

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
print(f"torch {torch.__version__}, triton {triton.__version__}, {torch.cuda.get_device_name()}")

## Solution

`%%writefile` saves the cell to `11_quantize_rowwise_int8.py`, the file to submit. Run it after every edit, then the check.

In [ ]:
%%writefile 11_quantize_rowwise_int8.py
import math

import torch
import triton
import triton.language as tl


@triton.autotune(
    # YOUR CODE HERE
)
@triton.jit
def _quantize_rowwise(x_ptr, output_ptr, output_maxs, n_elements,
                      BLOCK_SIZE: tl.constexpr, P2: tl.constexpr):
    # YOUR CODE HERE
    pass


def quantize_rowwise(x: torch.Tensor):
    # YOUR CODE HERE
    raise NotImplementedError

## Check

Compares with PyTorch on shapes with tails and checks some of the requirements. These are my tests, not the bot's: passing them doesn't guarantee the bot accepts the solution.

In [ ]:
PATH = "11_quantize_rowwise_int8.py"
spec = importlib.util.spec_from_file_location("sol", PATH)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)
SRC = open(PATH).read()

def quant_ref(x, denom):
    return torch.round(127.0 * x.float() / denom).clamp(-127, 127).to(torch.int8)


def assert_quant(q, ref):
    # int8 values may differ by 1 where rounding modes disagree on .5
    assert q.dtype == torch.int8, q.dtype
    assert q.shape == ref.shape, (q.shape, ref.shape)
    diff = (q.int() - ref.int()).abs()
    assert diff.max() <= 1, f"max |q - ref| = {diff.max().item()}"
    assert (diff > 0).float().mean() < 1e-3, f"{(diff > 0).sum().item()} elements off by 1"

for dtype in (torch.float32, torch.float16):
    x = torch.randn(33, 1000, device="cuda", dtype=dtype)
    q, absmaxs = sol.quantize_rowwise(x)
    ref_max = x.abs().amax(1)
    assert absmaxs.shape == (33,) and absmaxs.dtype == torch.float16, (absmaxs.shape, absmaxs.dtype)
    torch.testing.assert_close(absmaxs, ref_max.half())
    assert_quant(q, quant_ref(x, ref_max.float()[:, None]))
assert len(sol._quantize_rowwise.configs) >= 2, "the task asks for several autotune configs"

torch.cuda.synchronize()
print("PASS")

## Download

In [ ]:
from google.colab import files

files.download("11_quantize_rowwise_int8.py")